In [2]:
# ─── 1. IMPORTS ───────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from google.colab import drive

In [3]:
# ─── 3. LEITURA E IDENTIFICAÇÃO DAS PLANILHAS ─────────────────────────────────
# ⚙️  Ajuste PASTA_DRIVE para o caminho da pasta no seu Drive onde estão os CSVs.
# Exemplo: 'Meu Drive/dados_acidentes'  ou  'Meu Drive'  se estiverem na raiz.
PASTA_DRIVE = 'Meu Drive'   # ← altere aqui se necessário

ARQUIVOS_ESPERADOS = [
    '2023_famar.csv',
    '2024_famar.csv',
    '2023_fumes.csv',
    '2024_fumes.csv',
]

def identificar_instituicao(nome_arquivo: str) -> str:
    nome = nome_arquivo.lower()
    if 'fumes' in nome:
        return 'FUMES'
    elif 'famar' in nome:
        return 'FAMAR'
    raise ValueError(f'Não foi possível identificar a instituição em: {nome_arquivo}')

planilhas = {}   # {'nome_arquivo': DataFrame}

for nome in ARQUIVOS_ESPERADOS:
    caminho = f'{nome}'
    try:
        df = pd.read_csv(caminho)
    except FileNotFoundError:
        print(f'[AVISO] Arquivo não encontrado: {caminho}')
        continue

    df['instituicao'] = identificar_instituicao(nome)
    df['arquivo_origem'] = nome

    # Converter coluna de data (formato yyyy-dd-MM hh:mm:ss)
    # Tenta converter no formato com hora
    # Tenta converter no formato com hora
    datas_convertidas = pd.to_datetime(
        df['data_do_acidente'],
        format='%Y-%m-%d %H:%M:%S',
        errors='coerce'
    )

    # Onde falhou, tenta no formato sem hora
    datas_convertidas = datas_convertidas.fillna(
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d',
            errors='coerce'
        )
    )

    # Imprime datas inválidas
    mask_invalidas = (
        datas_convertidas.isna()
        & df['data_do_acidente'].notna()
    )

    if mask_invalidas.any():
        print('\nDatas com erro de conversão:')

        for idx, valor in df.loc[
            mask_invalidas,
            'data_do_acidente'
        ].items():

            print(
                f'  Linha {idx + 2}: '
                f'"{valor}" '
                f'(esperado: %Y-%m-%d %H:%M:%S '
                f'ou %Y-%m-%d)'
            )

    # Salva no dataframe
    df['data_do_acidente'] = datas_convertidas

    planilhas[nome] = df
    print(f'[OK] {nome} — {len(df)} registros | instituição: {df["instituicao"].iloc[0]}')

print(f'\nTotal de arquivos carregados: {len(planilhas)}')

[OK] 2023_famar.csv — 93 registros | instituição: FAMAR
[OK] 2024_famar.csv — 83 registros | instituição: FAMAR
[OK] 2023_fumes.csv — 9 registros | instituição: FUMES
[OK] 2024_fumes.csv — 12 registros | instituição: FUMES

Total de arquivos carregados: 4


In [4]:
# ─── 4. AGREGAÇÃO DOS BIÊNIOS ─────────────────────────────────────────────────
# Biênio FAMAR (2023 + 2024)
df_famar = pd.concat(
    [planilhas[f] for f in ['2023_famar.csv', '2024_famar.csv'] if f in planilhas],
    ignore_index=True
)

# Biênio FUMES (2023 + 2024)
df_fumes = pd.concat(
    [planilhas[f] for f in ['2023_fumes.csv', '2024_fumes.csv'] if f in planilhas],
    ignore_index=True
)

print(f'Biênio FAMAR — total de registros: {len(df_famar)}')
print(f'Biênio FUMES — total de registros: {len(df_fumes)}')

Biênio FAMAR — total de registros: 176
Biênio FUMES — total de registros: 21


In [5]:
import pandas as pd

FORMATO_DATA = '%Y-%m-%d %H:%M:%S'

print('=' * 60)
print('  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA')
print('=' * 60)

total_geral = 0

for nome, df in planilhas.items():
    n = len(df)
    total_geral += n

    inst = (
        df['instituicao'].iloc[0]
        if not df.empty and 'instituicao' in df.columns
        else 'N/D'
    )

    datas_nulas = df['data_do_acidente'].isna().sum()
    datas_invalidas = []

    for idx, val in df['data_do_acidente'].items():

        # Ignora nulos
        if pd.isna(val):
            continue

        linha_planilha = idx + 2  # +1 header +1 índice zero-based
        valor_original = str(val)

        # try:
        #     pd.to_datetime(
        #         valor_original,
        #         format=FORMATO_DATA,
        #         errors='raise'
        #     )

        # except Exception:

        #     motivo = (
        #         'Formato inválido. '
        #         'Esperado: yyyy-mm-dd HH:MM:SS '
        #         f'(ex.: 2023-05-31 00:00:00)'
        #     )

        #     datas_invalidas.append({
        #         'linha': linha_planilha,
        #         'valor': valor_original,
        #         'motivo': motivo
        #     })

    print(f'\nArquivo : {nome}')
    print(f'  Instituição       : {inst}')
    print(f'  Registros totais  : {n}')
    print(f'  Datas nulas       : {datas_nulas}')
    print(f'  Datas inválidas   : {len(datas_invalidas)}')

    if datas_invalidas:
        print('\n  Datas fora do formato:')

        print(
            f'  {"Linha":>6}  '
            f'{"Valor encontrado":<30}  '
            f'Motivo'
        )

        print(
            f'  {"-" * 6}  '
            f'{"-" * 30}  '
            f'{"-" * 60}'
        )

        for erro in datas_invalidas:
            print(
                f'  {erro["linha"]:>6}  '
                f'{erro["valor"]:<30}  '
                f'{erro["motivo"]}'
            )

    else:
        print('  Datas fora de formato : nenhuma')

print(f'\n{" TOTAL GERAL ":=^60}')
print(
    f'  {total_geral} registros importados '
    f'em {len(planilhas)} arquivos'
)

  RELATÓRIO DE IMPORTAÇÕES POR PLANILHA

Arquivo : 2023_famar.csv
  Instituição       : FAMAR
  Registros totais  : 93
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_famar.csv
  Instituição       : FAMAR
  Registros totais  : 83
  Datas nulas       : 1
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2023_fumes.csv
  Instituição       : FUMES
  Registros totais  : 9
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

Arquivo : 2024_fumes.csv
  Instituição       : FUMES
  Registros totais  : 12
  Datas nulas       : 0
  Datas inválidas   : 0
  Datas fora de formato : nenhuma

======================= TOTAL GERAL ========================
  197 registros importados em 4 arquivos


In [6]:
# ─── 5. FUNÇÃO: TRIMESTRE A PARTIR DA DATA ────────────────────────────────────
def adicionar_trimestre(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['trimestre'] = (
        pd.to_datetime(
            df['data_do_acidente'],
            format='%Y-%m-%d %H:%M:%S',
            errors='coerce'
        )
        .dt.quarter
        .apply(lambda x: f'T{x}' if pd.notna(x) else None)
    )

    return df


df_famar = adicionar_trimestre(df_famar)
df_fumes = adicionar_trimestre(df_fumes)

print(
    'Trimestres identificados — FAMAR:',
    sorted(df_famar['trimestre'].dropna().unique())
)

print(
    'Trimestres identificados — FUMES:',
    sorted(df_fumes['trimestre'].dropna().unique())
)

Trimestres identificados — FAMAR: ['T1.0', 'T2.0', 'T3.0', 'T4.0']
Trimestres identificados — FUMES: ['T1', 'T2', 'T3', 'T4']


In [7]:
# ─── 7. RELATÓRIO TRIMESTRAL DOS BIÊNIOS ──────────────────────────────────────
def relatorio_trimestral(df: pd.DataFrame, nome_inst: str):
    total = len(df)
    print(f'\n{" " + nome_inst + " — Biênio por Trimestre ":=^60}')
    print(f'  Total do biênio: {total} registros\n')

    por_trim = (
        df.groupby('trimestre', dropna=False)
          .size()
          .reset_index(name='n')
          .sort_values('trimestre')
    )

    print(f'  {"Trimestre":<15} {"N (abs)":>10} {"% do biênio":>14}')
    print(f'  {"-"*15} {"-"*10} {"-"*14}')

    for _, row in por_trim.iterrows():
        trim = str(row['trimestre']) if pd.notna(row['trimestre']) else 'Data inválida'
        n    = int(row['n'])
        pct  = (n / total * 100) if total > 0 else 0
        print(f'  {trim:<15} {n:>10} {pct:>13.1f}%')

    print(f'  {"TOTAL":<15} {total:>10} {100.0:>13.1f}%')

relatorio_trimestral(df_famar, 'FAMAR')
relatorio_trimestral(df_fumes, 'FUMES')


=============== FAMAR — Biênio por Trimestre ===============
  Total do biênio: 176 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1.0                    49          27.8%
  T2.0                    35          19.9%
  T3.0                    52          29.5%
  T4.0                    39          22.2%
  Data inválida            1           0.6%
  TOTAL                  176         100.0%

=============== FUMES — Biênio por Trimestre ===============
  Total do biênio: 21 registros

  Trimestre          N (abs)    % do biênio
  --------------- ---------- --------------
  T1                       5          23.8%
  T2                       6          28.6%
  T3                       7          33.3%
  T4                       3          14.3%
  TOTAL                   21         100.0%
